# 05 - Improvements and ablations (Part B-2 and Part D)

Three improvements, each with a motivation, each measured **individually on validation**, then combined:

| # | Side | Improvement | Motivation |
|---|------|-------------|------------|
| 1 | Data | Stronger augmentation (flip, scale jitter, colour jitter) | fights overfitting on an 8000-image subset |
| 2 | Data/loss | Class rebalancing (median-frequency weights inside the composite CE) | the dataset is heavily imbalanced (see 01) |
| 3 | Inference | Test-time augmentation (average prediction with the flipped image) | free accuracy at test time, no training |

The ablation table shows each one added **one at a time** and **cumulatively**, exactly as the brief asks.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q segmentation-models-pytorch transformers optuna thop joblib
print('installs done')

In [ ]:
import sys, os
import numpy as np
sys.path.insert(0, '/content/drive/MyDrive/coco_seg/project/code')
import config as C
import data, engine, losses, models, agent_search, report_utils
C.ensure_dirs()
print('GPU:', __import__('torch').cuda.get_device_name(0) if __import__('torch').cuda.is_available() else 'NONE - Runtime > Change runtime type > T4 GPU')
import torch
frames = data.load_frames()
CW = report_utils.load_result('class_weights')['median_frequency_weights']

def run_variant(name, tag, aug, balanced):
    """Train the composite config with one variation; resume-safe."""
    choice = report_utils.load_result(f'agent_weight_choice_{name}')
    train_loader = data.get_loader(frames['train'], 'train', batch_size=C.BATCH_SIZE[name], augment=aug)
    val_loader   = data.get_loader(frames['val'],   'val',   batch_size=C.BATCH_SIZE[name])
    model = models.build_model(name)
    comps = {k: losses.build_loss(k, C.NUM_CLASSES, CW if (balanced and k in ('ce', 'weighted_ce')) else None)
             for k in choice['components']}
    comp = losses.CompositeLoss(comps, choice['chosen_weights'])
    history = engine.train(model, f'{name}_{tag}', train_loader, val_loader,
                           comp, epochs=C.EPOCHS_MAIN, seed=C.SEEDS[0])
    val = engine.evaluate(model, val_loader)
    ck = os.path.join(C.CKPT_DIR, f'{name}_{tag}_final.pt')
    torch.save(model.state_dict(), ck)
    return model, val_loader, val, ck
print('helpers ready')

### Run the variants

Reference = the composite run from notebook 04 (augmentation ON, unbalanced, no TTA).

In [ ]:
for name in C.MODELS:
    abl = report_utils.load_result(f'ablations_{name}') or {}
    ref = report_utils.load_result(f'composite_main_{name}')
    abl['composite (reference: aug on)'] = ref['val']

    if 'improvement 1 OFF: no augmentation' not in abl:
        _, _, val, _ = run_variant(name, 'abl_noaug', aug=False, balanced=False)
        abl['improvement 1 OFF: no augmentation'] = val
    # improvement 2: class rebalancing
    if 'improvement 2: + class rebalancing' not in abl:
        _, _, val, _ = run_variant(name, 'abl_rebal', aug=True, balanced=True)
        abl['improvement 2: + class rebalancing'] = val
    # improvement 3: TTA (inference only, reuses the reference weights)
    if 'improvement 3: + TTA (hflip)' not in abl:
        model = models.build_model(name)
        model.load_state_dict(torch.load(os.path.join(C.CKPT_DIR, f'{name}_composite_final.pt'), map_location='cpu'))
        val_loader = data.get_loader(frames['val'], 'val', batch_size=C.BATCH_SIZE[name])
        abl['improvement 3: + TTA (hflip)'] = engine.evaluate(model, val_loader, tta=True)
        # cumulative: rebalanced + TTA
        model2 = models.build_model(name)
        model2.load_state_dict(torch.load(os.path.join(C.CKPT_DIR, f'{name}_abl_rebal_final.pt'), map_location='cpu'))
        abl['cumulative: rebalancing + TTA'] = engine.evaluate(model2, val_loader, tta=True)
    report_utils.save_result(f'ablations_{name}', abl)
    print(f'--- {name} ablations (val mIoU) ---')
    for k, v in abl.items():
        print(f'  {k:45s} {v["miou"]:.4f}')